# 16 — Sea-level-rise risk: SaTScan clusters → GeoAI story map

![PBIX-derived six perspectives](assets/six-thinking-hats-global-health.png)

This lab turns the SaTScan cluster summary embedded in `Prototype-July-8-2024.pbix` into a browser-first GeoAI investigation. The report’s `ALLMODELS` table supplies cluster rank, flood facilities, observed/expected ratio, relative risk, radius, and significance. Its `Climate-Sites-w-coords` table supplies city coordinates, elevation context, and scenario flood indicators.

This is a decision-support teaching example—not a new SaTScan run, a forecast, or a validated flood-hazard map.

## Start here: the question behind the code

**How can we explore the report's top ten clusters without mixing metrics?**

**What you will learn:** read → compare → animate → interpret. The numbered reading guides below follow the actual code cells. Read a guide, predict the output, then run its cell with **Shift+Enter**. Run from top to bottom; changing an upstream value requires rerunning the cells that use it.

**GeoAI connection:** Evidence interpretation supporting GeoAI: this notebook visualizes existing SaTScan results and fits no new model.

**Why this is useful:** Linked views help compare statistical risk, reported facilities and scenario indicators while exposing what additional evidence a decision needs.

**Where the work happens:** The small bundled PBIX derivative is read locally; Plotly shows bars, scenario bubbles and 3D points. GeoLibre maps representative city coordinates.

![Keep the risk axes honest. Start with ten cluster-ranked report rows and documented city aggregates. Display relative risk separately from original rank and facility counts. Compare A/B/C indicators as alternative scenarios, not as a forecast movie. Read 3D height as relative risk; it is not terrain or floodwater depth.](assets/learning/16/infographic.svg)

### Watch the idea unfold

[![Preview: Keep the risk axes honest — 32-second captioned walkthrough](assets/learning/16/poster.png)](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/files/assets/learning/16/watch.html)

[▶ Watch the captioned mini-video and read its transcript](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/files/assets/learning/16/watch.html) · [Launch this notebook](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/lab/index.html?path=16_sea_level_rise_sat_scan_geoai.ipynb) · [Python and browser learning guide](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/learning-guide.html)

The video is a silent four-step teaching storyboard with player controls and captions. Open the companion page to play, pause, seek or download it. The local copy is bundled under `assets/learning/16/`; downloaded media can be played offline. The infographic and the text below cover the same ideas without a video player.

**Video transcript:**

1. **Read (0–8 s):** Start with ten cluster-ranked report rows and documented city aggregates. Code landmark: `pd.read_csv(...)`.

2. **Compare (8–16 s):** Display relative risk separately from original rank and facility counts. Code landmark: `sort_values('relative_risk')`.

3. **Animate (16–24 s):** Compare A/B/C indicators as alternative scenarios, not as a forecast movie. Code landmark: `risk.melt(...)`.

4. **Interpret (24–32 s):** Read 3D height as relative risk; it is not terrain or floodwater depth. Code landmark: `z=risk.relative_risk`.


### Code guide 1 — Read the small report derivative

**Read the Python:** The browser setup installs the plotting and map tools. `read_csv` loads the bundled ten-row summary. Adding city and country strings creates a readable label without changing the original rank or risk fields.

**Look for:** Ten reported cluster summaries from the supplied PBIX, with supporting city-level aggregates.

**Pause and try:** Compare `all_facilities` and `n_sites`: these come from different table summaries and must not be treated as the same denominator.


In [ ]:
import sys
if sys.platform == 'emscripten':
    import micropip
    await micropip.install(['geolibre==3.0.0', 'pyodide-http', 'plotly==6.5.0'])
    import pyodide_http
    pyodide_http.patch_all()
from geolibre_lite import LiteMap as Map
import pandas as pd
import numpy as np
from IPython.display import display, Markdown
import plotly.express as px
import plotly.graph_objects as go
risk = pd.read_csv('data/pbix_top10_sat_scan.csv')
risk['city_label'] = risk['city_name'] + ', ' + risk['country']
risk


## Story map 1 — What does the PBIX evidence say?

SaTScan identifies unusually concentrated spatial risk. Rank and relative risk are evidence about reported clusters; they are not interchangeable with meters of water, population exposed, or a probability that a particular building floods.

### Code guide 2 — Compare rank, relative risk and location

**Read the Python:** Sorting the bar chart by relative risk changes display order, not the report's cluster rank. `itertuples` makes one representative city point per row, with properties for inspection. These points are not cluster boundaries.

**Look for:** Relative-risk bars colored by facility count and a 2D map of the ten reported clusters' representative city locations.

**Pause and try:** Identify the highest relative-risk bar. Is it also cluster rank 1 in the original report?


In [ ]:
fig = px.bar(risk.sort_values('relative_risk'), x='relative_risk', y='city_label', orientation='h', color='flood_facilities', color_continuous_scale='Turbo', labels={'relative_risk':'SaTScan relative risk (report units)','city_label':'City','flood_facilities':'Flood facilities'})
fig.update_layout(height=560, title='Reported SaTScan clusters: relative risk and flood-facility count')
fig.show()
m = Map(center=(35, 35), zoom=1.3, height='650px')
features = [{'type':'Feature','geometry':{'type':'Point','coordinates':[r.longitude,r.latitude]},'properties':{'city':r.city_label,'cluster_rank':int(r.cluster_rank),'relative_risk':float(r.relative_risk),'flood_facilities':int(r.flood_facilities)}} for r in risk.itertuples()]
m.add_geojson({'type':'FeatureCollection','features':features}, name='PBIX SaTScan top 10', radius=8, fillColor='#e45756', fillOpacity=0.8, strokeColor='#351c75', strokeWidth=1.5)
m


## Story map 2 — Scenario animation

The PBIX includes three scenario columns summarized here as A, B, and C. The animation is a sense-making device: it asks where the pattern changes most without pretending that a scenario value is direct water depth.

### Code guide 3 — Reshape columns for a scenario animation

**Read the Python:** `melt` turns the A/B/C columns into thirty rows: one city-scenario combination per row. A dictionary maps field names to scenario labels. `animation_frame` changes the displayed scenario while size and color encode the indicator.

**Look for:** A scenario-comparison animation. A/B/C labels represent alternatives, not a measured time sequence or rising water surface.

**Pause and try:** Check a changing bubble against the table. Why cannot its indicator value be read as metres of floodwater?


In [ ]:
scenario_cols = {'A':'flood_a_mean','B':'flood_b_mean','C':'flood_c_mean'}
long = risk.melt(id_vars=['city_label','relative_risk'], value_vars=list(scenario_cols.values()), var_name='scenario', value_name='scenario_indicator')
long['scenario'] = long['scenario'].map({v:k for k,v in scenario_cols.items()})
fig = px.scatter(long, x='relative_risk', y='city_label', animation_frame='scenario', size='scenario_indicator', color='scenario_indicator', color_continuous_scale='Viridis', hover_name='city_label', labels={'relative_risk':'SaTScan relative risk','city_label':'City','scenario_indicator':'PBIX scenario indicator'})
fig.update_layout(height=620, title='Animated scenario comparison — PBIX flood indicator, not water depth')
fig.show()


## Story map 3 — 3D risk and a responsible GeoAI deep dive

A 3D view can show several dimensions at once, but it can also create false confidence. Height encodes relative risk, point size encodes facilities, and color encodes the all-scenario indicator. Always compare it with the table and 2D map.

### Code guide 4 — Read the three-dimensional axes literally

**Read the Python:** Longitude, latitude and relative risk become x, y and z. `np.clip` bounds marker sizes for legibility, so size is only a display cue. Color uses the all-scenario mean from the supporting sites table.

**Look for:** A rotatable 3D statistical scatter; the vertical axis is relative risk, not terrain elevation or water depth.

**Pause and try:** Compare the 3D view with the bars before drawing a conclusion; the camera can obscure points.


In [ ]:
fig3d = go.Figure(go.Scatter3d(x=risk.longitude, y=risk.latitude, z=risk.relative_risk, mode='markers+text', text=risk.city_name, textposition='top center', marker={'size':np.clip(risk.flood_facilities/18,6,18),'color':risk.flood_all_mean,'colorscale':'Viridis','showscale':True,'colorbar':{'title':'All-scenario indicator'}}, hovertemplate='%{text}<br>relative risk=%{z:.1f}<extra></extra>'))
fig3d.update_layout(height=680, title='3D sense-making view: location × relative risk × reported facilities', scene={'xaxis_title':'Longitude','yaxis_title':'Latitude','zaxis_title':'Relative risk'})
fig3d.show()


## Six perspectives for the next decision

- **White / evidence:** Which source field, date, denominator, and spatial unit support each claim?
- **Red / experience:** What would residents, responders, and health workers notice that the table cannot show?
- **Black / caution:** Where could missing data, geocoding, model assumptions, or map design harm a community?
- **Yellow / opportunity:** Which early-warning, planning, or service-access decision could this evidence improve?
- **Green / alternatives:** What other data, methods, or community-led interpretations should challenge the first model?
- **Blue / process:** Who owns the next action, what gets validated, and when do we stop using the model?
- **Purple / synthesis:** What is the smallest accountable next step that preserves disagreement and makes uncertainty visible?

A useful GeoAI workflow is **discover → join → engineer → compare → explain → challenge**. The PBIX SaTScan result identifies reported spatial clusters; the scenario fields provide a comparative teaching layer. The next responsible step is validation with local data and accountable partners—not a stronger claim.

## Data & software citations

- User-supplied Power BI report: `Prototype-July-8-2024.pbix`. The committed CSV is a derived top-10 summary, not the complete report model.
- SaTScan-derived report table: `ALLMODELS`; supporting table: `Climate-Sites-w-coords`.
- [GeoLibre](https://geolibre.app/), [GeoAI](https://opengeoai.org/), and [Plotly Python](https://plotly.com/python/).
- Six Thinking Hats concepts are associated with Edward de Bono; the purple synthesis framing and supplied graphic are retained as teaching context.
- Preserve the report’s original source citations and licensing before publishing any derivative result.

## Make sense of the result

![A taller point means deeper flooding? Here the z axis is a statistical relative risk. I will read the axes and check the source denominators.](assets/learning/16/cartoon.svg)

**Cartoon transcript:** A taller point means deeper flooding? → Here the z axis is a statistical relative risk. → I will read the axes and check the source denominators.

**Explain it to a colleague:** Why should the two facility/site counts not be interchanged?

<details><summary>Check your reasoning</summary>

ALLMODELS cluster counts and supporting city-site aggregates use different selections; their denominators need separate interpretation.

</details>

Write three sentences: **I used…** (source and units), **the code computed…** (method), and **I can conclude…** (plus a remaining question). Point to the map, chart or table that supports each claim.

All new teaching illustrations are original schematics. Their marks and fictional dialogue are explanatory, not observations, fitted predictions or captured AI conversations. The existing citation cells identify the analytical data.
